# Customer Churn Analysis — Telco Dataset

Business data analysis project exploring customer churn behavior for a
telecommunications company. This is **Phase 1** of an end-to-end churn
analysis and prediction project. The Machine Learning phase will be added
in a future update.

## Table of Contents

**Phase 1 — Business & Data Understanding (this project)**
1. Introduction
2. Business Understanding
3. Data Understanding
4. Data Cleaning *(`02_data_cleaning.ipynb`)*
5. Exploratory Data Analysis *(`03_eda.ipynb`)*
6. Statistical Analysis *(`04_statistical_analysis.ipynb`)*
7. Feature Engineering *(`05_feature_engineering.ipynb`)*
8. Key Findings *(`05_feature_engineering.ipynb`)*
9. Conclusion *(`05_feature_engineering.ipynb`)*

**Phase 2 — Machine Learning (future update, `06_modeling.ipynb` / `07_model_interpretation.ipynb`)**

10. Preprocessing
11. Baseline Model
12. ML Models
13. Model Evaluation
14. Imbalanced Data Handling
15. Model Interpretation
16. Business-Oriented Evaluation
17. Customer Risk Scoring
18. Retention Strategy
19. Final Dashboard / Visual Summary
20. Final Business Recommendations

# 1. Introduction

Customer churn — when a customer stops doing business with a company — is one
of the most critical metrics for subscription-based businesses such as
telecommunications providers. Acquiring a new customer is significantly more
expensive than retaining an existing one, making churn analysis a high-impact
area for business decision-making.

This project analyzes the Telco Customer Churn dataset to understand churn
behavior, identify its key drivers, and lay the groundwork for a predictive
model in a future phase.

**Dataset:** [Telco Customer Churn](https://www.kaggle.com/datasets/blastchar/telco-customer-churn)
— customer-level data including demographics, account information, subscribed
services, billing details, and churn status.

**Project Phase:** Business Understanding through Feature Engineering
(Phase 1 of 2). The Machine Learning phase will be added in a future update.

# 2. Business Understanding

## 2.1 Goal

- Understand the overall churn rate and how it varies across customer segments
- Identify the factors most strongly associated with churn
- Identify characteristics of high-risk customers
- Provide a data-driven foundation for retention strategy recommendations
- Prepare clean, well-engineered features for churn prediction modeling (Phase 2)

## 2.2 Business Questions

- What is the overall churn rate?
- Which types of customers churn the most?
- Which factors are most strongly associated with churn?
- Does contract type affect churn?
- Does tenure affect churn?
- Does a higher monthly charge lead to more churn?
- Which services are associated with higher retention?
- Is payment method associated with churn?
- Can churn be predicted before it happens? *(addressed in Phase 2)*

# 3. Data Understanding

#### Objective

This section loads the raw dataset and establishes a baseline understanding
of its structure, quality, and target variable before any cleaning or
transformation is performed.

## 3.1 Load Data

In [1]:
import pandas as pd
import numpy as np

data_path = "../data/raw/"

df = pd.read_csv(data_path + "WA_Fn-UseC_-Telco-Customer-Churn.csv")

df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 3.2 Dataset Overview

#### Objective

A first look at the dataset's size and structure.

In [2]:
df.shape

(7043, 21)

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


## 3.3 Data Dictionary

#### Objective

Documenting each column's meaning and data type provides a reference for the
rest of the analysis and speeds up onboarding for anyone reviewing this
project.

In [4]:
data_dictionary = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "n_unique": [df[col].nunique() for col in df.columns],
    "sample_value": [df[col].dropna().iloc[0] if df[col].notna().any() else None for col in df.columns]
})

data_dictionary

,column,dtype,n_unique,sample_value
0,customerID,object,7043,7590-VHVEG
1,gender,object,2,Female
2,SeniorCitizen,int64,2,0
3,Partner,object,2,Yes
4,Dependents,object,2,No
5,tenure,int64,73,1
6,PhoneService,object,2,No
7,MultipleLines,object,3,No phone service
8,InternetService,object,3,DSL
9,OnlineSecurity,object,3,No


#### Column Descriptions

| Column | Description |
|---|---|
| `customerID` | Unique customer identifier |
| `gender` | Customer gender |
| `SeniorCitizen` | Whether the customer is a senior citizen (1/0) |
| `Partner` | Whether the customer has a partner |
| `Dependents` | Whether the customer has dependents |
| `tenure` | Number of months the customer has stayed with the company |
| `PhoneService`, `MultipleLines` | Phone service subscription details |
| `InternetService` | Type of internet service (DSL / Fiber optic / No) |
| `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies` | Subscribed add-on services |
| `Contract` | Contract term (Month-to-month / One year / Two year) |
| `PaperlessBilling` | Whether the customer uses paperless billing |
| `PaymentMethod` | Payment method used |
| `MonthlyCharges` | Current monthly charge amount |
| `TotalCharges` | Total amount charged to the customer (stored as text — see Data Cleaning) |
| `Churn` | **Target variable** — whether the customer churned (Yes/No) |

## 3.4 Missing Values

#### Objective

Checking for missing values using pandas' standard null detection. Note that
this may not catch all cases — `TotalCharges` is expected to contain blank
strings rather than true `NaN` values for new customers, which will only
surface after the type conversion in Data Cleaning.

In [6]:
df.isnull().sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

## 3.5 Duplicates

#### Objective

Checking for fully duplicated rows and duplicated `customerID` values, since
each row should represent one unique customer.

In [7]:
print("Fully duplicated rows:", df.duplicated().sum())
print("Duplicated customerID:", df["customerID"].duplicated().sum())

Fully duplicated rows: 0
Duplicated customerID: 0


## 3.6 Unique Values & Cardinality

#### Objective

Reviewing the unique values of categorical columns to understand their
cardinality and check for inconsistent labels (e.g. typos, unexpected
categories) before cleaning.

In [8]:
categorical_cols = df.select_dtypes(include="object").columns.drop("customerID")

for col in categorical_cols:
    print(f"{col}: {df[col].unique()}")
    print()

gender: ['Female' 'Male']

Partner: ['Yes' 'No']

Dependents: ['No' 'Yes']

PhoneService: ['No' 'Yes']

MultipleLines: ['No phone service' 'No' 'Yes']

InternetService: ['DSL' 'Fiber optic' 'No']

OnlineSecurity: ['No' 'Yes' 'No internet service']

OnlineBackup: ['Yes' 'No' 'No internet service']

DeviceProtection: ['No' 'Yes' 'No internet service']

TechSupport: ['No' 'Yes' 'No internet service']

StreamingTV: ['No' 'Yes' 'No internet service']

StreamingMovies: ['No' 'Yes' 'No internet service']

Contract: ['Month-to-month' 'One year' 'Two year']

PaperlessBilling: ['Yes' 'No']

PaymentMethod: ['Electronic check' 'Mailed check' 'Bank transfer (automatic)'
 'Credit card (automatic)']

TotalCharges: ['29.85' '1889.5' '108.15' ... '346.45' '306.6' '6844.5']

Churn: ['No' 'Yes']



## 3.7 Target Definition & Class Distribution

#### Objective

This project's target variable is `Churn`, indicating whether a customer
discontinued service. Understanding its class balance is essential, since a
skewed distribution affects both the interpretation of EDA results and the
modeling strategy in Phase 2 (e.g. need for class weighting or resampling).

In [9]:
churn_distribution = df["Churn"].value_counts()
churn_percentage = df["Churn"].value_counts(normalize=True) * 100

pd.DataFrame({
    "count": churn_distribution,
    "percentage": churn_percentage.round(2)
})

,count,percentage
Churn,,
No,5174,73.46
Yes,1869,26.54


#### Observation

The dataset is expected to show a class imbalance, with the "No" (non-churn)
class representing the majority. This imbalance is an important
consideration for the Statistical Analysis (Section 6) and, later, for model
training and evaluation in Phase 2, where accuracy alone would be a
misleading metric.

#### Summary

The dataset contains one row per customer with demographic, account, service,
and billing information, alongside the `Churn` target. No duplicate customers
were found. `TotalCharges` is stored as text and will require conversion, and
the target variable shows class imbalance that will inform the analysis and
modeling approach going forward. Data Cleaning continues in
`02_data_cleaning.ipynb`.